In [ ]:
%matplotlib widget

In [ ]:
import hyperspy.api as hs
import pyxem as pxm
import matplotlib.pyplot as plt
import numpy as np

from scipy.interpolate import RegularGridInterpolator
from scipy.optimize import root, least_squares

from scipy.ndimage import shift
from skimage.transform import AffineTransform, warp

from pathlib import Path

import importlib
import get_transforms
import get_masks
importlib.reload(get_transforms)
importlib.reload(get_masks)

In [ ]:
#funksjon som laster inn zspy-fil og summerer diffrkasjonsmønsteret --> brightfield-bilde
def process_stem(file_path):
    s = hs.load(file_path, lazy=True)
    s_bs = s.sum(axis=(-1, -2))
    s_bs.compute()
    return s_bs, s

#bruker funksjonen på alle zspy-filene fra 002 - 023 = 22 filer
data_folder = Path("data")
files = sorted(data_folder.glob("*.zspy"))
original_images = {}
s_dir = {}

for file in files:
    s_bs, s = process_stem(file)
    original_images[file.stem] = s_bs
    s_dir[file.stem] = s

#Sjekker om filene er prosessert: 
print(f"Processed {len(original_images)} files")

In [ ]:
registered_images, transformations, shifts = get_transforms.main(original_images)

In [ ]:
# Mask is not really dependent on transformation, 002 is just normalized
# Center is (row, column)
masks, center, radii = get_masks.main(original_images, registered_images)

In [ ]:
def find_root(data, mask):
    rows, cols = np.arange(data.shape[0]), np.arange(data.shape[1])
    field = np.stack((data.real, data.imag), axis=-1)
    f = RegularGridInterpolator((rows, cols), field)
    
    valid = mask[:-1, :-1] & mask[1:, :-1] & mask[:-1, 1:] & mask[1:, 1:]
    
    corners = np.stack((
        data[:-1, :-1], data[1:, :-1],
        data[:-1, 1:], data[1:, 1:]
    ))
    
    candidates = valid & (
        (corners.real.min(0) <= 0) & (corners.real.max(0) >= 0) &
        (corners.imag.min(0) <= 0) & (corners.imag.max(0) >= 0)
    )
    
    solutions = []
    for r, c in np.argwhere(candidates):
        sol = least_squares(
            lambda p: f(p).ravel(),
            (r + 0.5, c + 0.5),
            bounds=([r, c], [r + 1, c + 1])
        )
        solutions.append(sol)
    
    if not solutions:
        return (None, None)
        # raise RuntimeError("No zero-crossing candidate found inside the mask")
    
    result = min(solutions, key=lambda s: np.linalg.norm(s.fun))
    
    # print("row, col:", result.x)
    # print("x, y:", result.x[::-1])
    # print("residual:", np.linalg.norm(result.fun))
    return result.x

In [ ]:
def rotate_data(data):
    data = (
        data.isig[0].data
        + 1j * data.isig[1].data
    )
    deg = 30 # Kanskje 90 grader feil
    rot = np.exp(1j * np.deg2rad(deg))
    data = data * rot
    return np.stack((data.real, data.imag), axis=-1)

In [ ]:
def warp_data(data, transformation, shift_vec):
    data_warp = warp(
        data,
        inverse_map=AffineTransform(matrix = transformation.params),
        preserve_range=True
    )
    
    data_shift_x = shift(
        data_warp[:, :, 0],
        shift_vec
    )
    
    data_shift_y = shift(
        data_warp[:, :, 1],
        shift_vec
    )
    
    return (
        data_shift_x
        + 1j * data_shift_y
    )

In [ ]:
# def normalize_imag(data, mask):
#     def normalize(data):
#         axis = (-2, -1)
#         data = data-data[mask].min()
#         data = data/data[mask].max()
#         return data*2-1
#     real_norm = normalize(data.real)
#     imag_norm = normalize(data.imag)*1j
#     return real_norm+imag_norm

In [ ]:
s_bs_corr_dir = {}
for s_name in s_dir:
    s = s_dir[s_name]
    s_bs = s.get_direct_beam_position(method="center_of_mass")
    s_bs.compute()
    s_bs_lp = s_bs.get_linear_plane()
    s_bs_corr_dir[s_name] = s_bs-s_bs_lp

In [ ]:
result = [[(None, None) for _ in range(len(s_dir))] for _ in range(len(masks))]

for s_name in s_dir:
    print(s_name)
    s = s_dir[s_name]
    for i, mask in enumerate(masks):
        data = rotate_data(s_bs_corr_dir[s_name])
        data = warp_data(data, transformations[s_name], shifts[s_name])
        data_masked = data*mask
        # data = normalize_imag(data_masked, mask)

        # Janky hack mate
        result[i][int(s_name)-2] = find_root(data, mask)

# Chat plotting for inspection

In [ ]:
from matplotlib.patches import Circle
import numpy as np
import matplotlib.pyplot as plt

# Select vortex using zero-based indexing.
vortex_idx = 5

# Keep the same ordering used when result was created:
# 002, 003, ..., 023
image_names = sorted(s_dir.keys(), key=int)

mask_center_rc = np.asarray(center[vortex_idx], dtype=float)
mask_radius = float(radii[vortex_idx])

row0, col0 = mask_center_rc

# Crop around the selected vortex.
half_width = max(14, int(np.ceil(1.8 * mask_radius)))

rmin = max(0, int(np.floor(row0 - half_width)))
rmax = min(masks[vortex_idx].shape[0],
           int(np.ceil(row0 + half_width + 1)))

cmin = max(0, int(np.floor(col0 - half_width)))
cmax = min(masks[vortex_idx].shape[1],
           int(np.ceil(col0 + half_width + 1)))

# Generate registered x-shift images only for visualization.
# The magnetic centers are read directly from result.
x_shift_images = {}

for s_name in image_names:
    vector_data = rotate_data(s_bs_corr_dir[s_name])

    registered_field = warp_data(
        vector_data,
        transformations[s_name],
        shifts[s_name]
    )

    x_shift_images[s_name] = registered_field.real

In [ ]:
n_images = len(image_names)
ncols = 6
nrows = int(np.ceil(n_images / ncols))

# Determine common grayscale limits across all displayed crops.
crop_values = []

for s_name in image_names:
    crop = x_shift_images[s_name][rmin:rmax, cmin:cmax]
    finite = crop[np.isfinite(crop)]

    if finite.size > 0:
        crop_values.append(finite)

if crop_values:
    crop_values = np.concatenate(crop_values)
    display_limit = np.nanpercentile(np.abs(crop_values), 98)

    if not np.isfinite(display_limit) or display_limit == 0:
        display_limit = 1.0
else:
    display_limit = 1.0

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(3.2 * ncols, 3.2 * nrows),
    constrained_layout=True
)

axes = np.asarray(axes).ravel()
last_im = None

for image_idx, (ax, s_name) in enumerate(zip(axes, image_names)):
    x_crop = x_shift_images[s_name][rmin:rmax, cmin:cmax]

    last_im = ax.imshow(
        x_crop,
        cmap="gray",
        origin="upper",
        vmin=-display_limit,
        vmax=display_limit,
        interpolation="nearest"
    )

    # The cyan circle shows the structural mask.
    circle = Circle(
        (col0 - cmin, row0 - rmin),
        radius=mask_radius,
        fill=False,
        color="cyan",
        linewidth=1.2
    )
    ax.add_patch(circle)

    # Use the previously calculated center.
    root_rc = result[vortex_idx][image_idx]

    valid_root = (
        root_rc is not None
        and len(root_rc) == 2
        and root_rc[0] is not None
        and root_rc[1] is not None
        and np.all(np.isfinite(root_rc))
    )

    if valid_root:
        root_row, root_col = root_rc

        # Matplotlib expects x=column and y=row.
        ax.plot(
            root_col - cmin,
            root_row - rmin,
            marker="+",
            color="red",
            markersize=13,
            markeredgewidth=2.2
        )

        ax.plot(
            root_col - cmin,
            root_row - rmin,
            marker="o",
            markerfacecolor="none",
            markeredgecolor="red",
            markersize=9,
            markeredgewidth=1.2
        )

        ax.set_title(
            f"{s_name}\n"
            f"x={root_col:.2f}, y={root_row:.2f}",
            fontsize=9
        )
    else:
        ax.set_title(
            f"{s_name}\nNO ROOT",
            fontsize=9,
            color="red"
        )

    ax.set_xticks([])
    ax.set_yticks([])

# Hide the two unused panels in the 4 x 6 grid.
for ax in axes[n_images:]:
    ax.axis("off")

fig.suptitle(
    f"Vortex {vortex_idx}: stored magnetic center on x-shift",
    fontsize=15
)

if last_im is not None:
    cbar = fig.colorbar(
        last_im,
        ax=axes[:n_images].tolist(),
        shrink=0.75,
        pad=0.015
    )
    cbar.set_label("x-shift")

plt.show()

# Lage god figur

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.colors import hsv_to_rgb
from matplotlib.patches import Circle


# ============================================================
# SETTINGS
# ============================================================

# Zero-based vortex index.
vortex_idx = 16

# Match the example figure: 4 rows x 5 columns = first 20 images.
nrows, ncols = 4, 5

# Direction-color rotation, used only to orient the color wheel.
color_rotation_deg = 135

# Set this if you want to save the figure.
save_figure = False
output_filename = "magnetic_vortex_tracking_simple.png"


# ============================================================
# PREPARE THE REGISTERED COMPLEX SHIFT FIELDS
# ============================================================

# Keep the same image order as the root calculation:
# 002, 003, ..., 023
image_names = sorted(s_dir.keys(), key=int)

# The requested compact figure contains the first 20 images.
plot_names = image_names[:nrows * ncols]

registered_fields_simple = {}

for s_name in plot_names:
    # Uses exactly the old processing logic.
    vector_data = rotate_data(s_bs_corr_dir[s_name])

    registered_fields_simple[s_name] = warp_data(
        vector_data,
        transformations[s_name],
        shifts[s_name]
    )


# ============================================================
# DEFINE THE CROP
# ============================================================

row0, col0 = np.asarray(center[vortex_idx], dtype=float)
radius = float(radii[vortex_idx])

half_width = max(14, int(np.ceil(1.8 * radius)))

first_shape = registered_fields_simple[plot_names[0]].shape

rmin = max(0, int(np.floor(row0 - half_width)))
rmax = min(first_shape[0], int(np.ceil(row0 + half_width + 1)))

cmin = max(0, int(np.floor(col0 - half_width)))
cmax = min(first_shape[1], int(np.ceil(col0 + half_width + 1)))

fields = [
    registered_fields_simple[name][rmin:rmax, cmin:cmax]
    for name in plot_names
]


# ============================================================
# USE ONE MAGNITUDE SCALE FOR ALL PANELS
# ============================================================

finite_magnitudes = [
    np.abs(field[np.isfinite(field)]).ravel()
    for field in fields
    if np.any(np.isfinite(field))
]

if finite_magnitudes:
    all_magnitudes = np.concatenate(finite_magnitudes)
    magnitude_limit = np.nanpercentile(all_magnitudes, 98)

    if not np.isfinite(magnitude_limit) or magnitude_limit <= 0:
        magnitude_limit = 1.0
else:
    magnitude_limit = 1.0


def direction_rgb(field, rotation_deg=135):
    """
    Convert a complex shift field into an RGB image.

    Hue:
        Direction of the complex vector.

    Brightness:
        Magnitude of the complex vector.
    """
    angle = np.angle(field) + np.deg2rad(rotation_deg)
    hue = np.mod(angle, 2 * np.pi) / (2 * np.pi)

    value = np.clip(
        np.abs(field) / magnitude_limit,
        0,
        1
    )

    # Make low-magnitude regions visible while preserving contrast.
    value = 0.18 + 0.82 * np.sqrt(value)

    hsv = np.stack(
        (
            hue,
            np.ones_like(hue),
            value
        ),
        axis=-1
    )

    rgb = hsv_to_rgb(hsv)
    rgb[~np.isfinite(field)] = 0

    return rgb


# ============================================================
# CREATE THE FIGURE
# ============================================================

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(8, 6.4),
    sharex=True,
    sharey=True
)

axes = np.asarray(axes).ravel()

for image_idx, (ax, s_name, field) in enumerate(
    zip(axes, plot_names, fields)
):
    ax.imshow(
        direction_rgb(
            field,
            rotation_deg=color_rotation_deg
        ),
        origin="upper",
        interpolation="nearest"
    )

    # Original structural vortex mask.
    ax.add_patch(
        Circle(
            (col0 - cmin, row0 - rmin),
            radius=radius,
            fill=False,
            color="cyan",
            linewidth=1.0
        )
    )

    # Old main.ipynb stores roots as:
    # result[vortex index][image index] = (row, column)
    root_rc = result[vortex_idx][image_idx]

    valid_root = (
        root_rc is not None
        and len(root_rc) == 2
        and root_rc[0] is not None
        and root_rc[1] is not None
        and np.all(np.isfinite(root_rc))
    )

    if valid_root:
        root_row, root_col = np.asarray(root_rc, dtype=float)

        # Convert full-image coordinates to crop coordinates.
        x = root_col - cmin
        y = root_row - rmin

        # White ring remains visible across the color wheel.
        ax.plot(
            x,
            y,
            marker="o",
            markersize=7,
            markerfacecolor="none",
            markeredgecolor="white",
            markeredgewidth=1.5
        )

        # Black plus identifies the fitted root precisely.
        ax.plot(
            x,
            y,
            marker="+",
            markersize=7,
            color="black",
            markeredgewidth=1.5
        )

    ax.set_axis_off()


# Hide unused panels if the layout is changed later.
for ax in axes[len(fields):]:
    ax.set_axis_off()

plt.subplots_adjust(
    left=0,
    right=1,
    bottom=0,
    top=1,
    wspace=0.02,
    hspace=0.02
)

if save_figure:
    fig.savefig(
        output_filename,
        dpi=300,
        bbox_inches="tight",
        pad_inches=0
    )

plt.show()

In [ ]:
plt.savefig("magnetic_vortex_tracking.png", dpi=300, bbox_inches="tight", pad_inches=0)